# PieceSpice — Notebook 04a: Simple Switching Subset (M4)

**Test:** gate pulse at **50 kHz / 50 % duty**, resistive load sized
for **1 A** on-state current. The vendor model and our synthesized
Tier-2 CPWL macromodel are simulated under **identical** conditions;
their waveforms and extracted metrics are compared.

- Vendor: set `VENDOR` in the setup cell (any configured vendor)
- Core solvers: `simulation/testbenches/simple_switching.py` +
  `metrics/fitness.py` — this notebook only calls and plots.

In [1]:
# PieceSpice — Notebook 04a: Simple Switching Subset (M4)
# 50 kHz / 50 % duty, resistive load sized for the YAML target current.
# All computation lives in library modules; this notebook only calls & plots.

import sys, os
from pathlib import Path
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

VENDOR = "epc2216"          # change to any configured vendor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from common.plotting import setup_style

from simulation.testbenches.simple_switching import (
    run_simple_switching, steady_state_slice, N_PERIODS_STEADY)
from metrics.fitness import (compare_switching_fitness, _metrics_from_run,
                             infer_drive_conditions)
from pipeline.fitting import fit_pwl_model

cfg = yaml.safe_load(
    Path(f"config/inputs/{VENDOR}.yaml").read_text(encoding="utf-8"))

# --- switching conditions (subset defaults; override here if needed) ---
VBUS = float(cfg["conditions"].get("Vbus",
            cfg["conditions"].get("Vds_fixed", 10.0)))
VGS_ON = float(cfg["conditions"]["Vgs_fixed"])
I_LOAD = 1.0                # A - user spec: total system current 1 A
TEMP = float(cfg["conditions"].get("temperature", 25.0))
# Both models are live-simulated here, so the drive is self-consistent. When
# comparing against a SAVED golden CSV, recover its conditions with
# infer_drive_conditions() instead of using today's YAML.

print(f"Vendor : {VENDOR} ({cfg['vendor']})")
print(f"Vbus   : {VBUS} V | Vgs_on: {VGS_ON} V | I_load: {I_LOAD} A")

# plotting helper (house rule: legend EAST outside)
def _ax_legend_east(ax):
    leg = ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
    return leg

setup_style()

Vendor : epc2216 (epc)
Vbus   : 12.0 V | Vgs_on: 5.0 V | I_load: 1.0 A


---
## 1 · Run Both Models

In [ ]:
# --- Run both models through the identical testbench ---
REBUILD_TIER2 = False   # set True to force re-fitting via notebook 02 logic

t2_path = Path("reports/models") / f"{VENDOR}_PIECESPICE_T2.lib"

if REBUILD_TIER2 or not t2_path.exists():
    if not t2_path.exists():
        raise FileNotFoundError(
            f"Tier-2 model for '{VENDOR}' not found at {t2_path}.\n"
            "Run notebooks/02_piecewise_approximation.ipynb first to fit and "
            "synthesize it (or set REBUILD_TIER2 = True in this cell).")
    tier2_lib = fit_pwl_model(VENDOR)
else:
    tier2_lib = t2_path
print("Tier-2 lib:", tier2_lib)

run_golden = run_simple_switching(
    Path(cfg["library"]), cfg["subcircuit"], cfg["pins"]["pin_order"],
    vbus=VBUS, vgs_on=VGS_ON, i_load_target=I_LOAD, temperature=TEMP)

run_tier2 = run_simple_switching(
    Path(tier2_lib), tier2_lib.stem, ["drain", "gate", "source"],
    vbus=VBUS, vgs_on=VGS_ON, i_load_target=I_LOAD, temperature=TEMP)

print("golden :", len(run_golden.time), "points")
print("tier-2 :", len(run_tier2.time), "points")

Tier-2 lib: reports\models\epc2216_PIECESPICE_T2.lib


Exception ignored from cffi callback <function NgSpiceShared._send_char at 0x000001932092B380>:
Traceback (most recent call last):
  File "c:\Users\GNKRISHNAN\Documents\GitHub\PieceSpice\.venv\Lib\site-packages\PySpice\Spice\NgSpice\Shared.py", line 608, in _send_char
    @staticmethod
KeyboardInterrupt: 


---
## 2 · Full Switching Waveform — 3 Cycles Overlay

In [ ]:
# --- Comparison plot 1: full run — 3 steady switching cycles ---
fig, ax = plt.subplots(figsize=(11.5, 6.0), layout="constrained")
ax.plot(run_golden.time * 1e6, run_golden.v_ds, lw=1.6,
        color="tab:blue", label="Vendor golden")
ax.plot(run_tier2.time * 1e6, run_tier2.v_ds, lw=1.4, ls="--",
        color="tab:red", alpha=0.9, label="Tier-2 CPWL")
ax.set_xlabel("Time [µs]")
ax.set_ylabel("Vds [V]")
ax.set_title(f"Switching Transient — {VENDOR} "
             f"(50 kHz, 50% duty, resistive {I_LOAD:.0f} A)")
_ax_legend_east(ax)
plt.show()

---
## 3 · Zoomed View — One Cycle (Turn-On / Turn-Off Edges)

In [ ]:
# --- Comparison plot 2: zoomed on ONE turn-on + turn-off of the last cycle ---
period = 1.0 / 50_000.0
t_end = run_golden.time[-1]
t_lo = t_end - period                      # last full cycle
mask_g = (run_golden.time >= t_lo) & (run_golden.time <= t_end)
mask_t = (run_tier2.time >= t_lo) & (run_tier2.time <= t_end)

fig, axs = plt.subplots(2, 1, figsize=(11.5, 8.0), sharex=False,
                        layout="constrained")

# turn-on window (first ~15% of the cycle)
on_lo, on_hi = t_lo, t_lo + 0.15 * period
axs[0].plot(run_golden.time[mask_g] * 1e6,
            run_golden.v_ds[mask_g], lw=1.8, color="tab:blue",
            label="Vendor golden")
axs[0].plot(run_tier2.time[mask_t] * 1e6,
            run_tier2.v_ds[mask_t], lw=1.5, ls="--", color="tab:red",
            label="Tier-2 CPWL")
axs[0].set_xlim(on_lo * 1e6, on_hi * 1e6)
axs[0].set_ylabel("Vds [V]")
axs[0].set_title("Zoom: Turn-On Edge")
_ax_legend_east(axs[0])

# turn-off window (last ~15% of the cycle)
off_lo, off_hi = t_end - 0.15 * period, t_end
axs[1].plot(run_golden.time[mask_g] * 1e6,
            run_golden.v_ds[mask_g], lw=1.8, color="tab:blue",
            label="Vendor golden")
axs[1].plot(run_tier2.time[mask_t] * 1e6,
            run_tier2.v_ds[mask_t], lw=1.5, ls="--", color="tab:red",
            label="Tier-2 CPWL")
axs[1].set_xlim(off_lo * 1e6, off_hi * 1e6)
axs[1].set_xlabel("Time [µs]")
axs[1].set_ylabel("Vds [V]")
axs[1].set_title("Zoom: Turn-Off Edge")
_ax_legend_east(axs[1])
plt.show()

---
## 4 · Fitness Metrics

In [ ]:
# --- Fitness metrics table ---
fit = compare_switching_fitness(
    vbus=VBUS, vgs_on=VGS_ON, i_load_target=I_LOAD, temperature=TEMP,
    golden_run=run_golden, tier2_run=run_tier2)   # reuse section-1 runs

mg, mt = fit.golden, fit.tier2
df = pd.DataFrame({
    "metric": ["td_on [ns]", "tr [ns]", "td_off [ns]", "tf [ns]",
               "E_on [µJ]", "E_off [µJ]",
               "Vds on-state [mV]", "Id peak [A]",
               "waveform rel-RMSE [%]"],
    "vendor golden": [mg.td_on_ns, mg.tr_ns, mg.td_off_ns, mg.tf_ns,
                      mg.e_on_uj, mg.e_off_uj,
                      mg.vds_on_mean_v * 1e3, mg.i_peak_a, np.nan],
    "tier-2 cpwl": [mt.td_on_ns, mt.tr_ns, mt.td_off_ns, mt.tf_ns,
                    mt.e_on_uj, mt.e_off_uj,
                    mt.vds_on_mean_v * 1e3, mt.i_peak_a,
                    fit.waveform_rel_rmse_pct],
})
errs = list(fit.timing_rel_err_pct.values()) + \
       list(fit.energy_rel_err_pct.values()) + [np.nan]
df["error %"] = [round(v, 2) if np.isfinite(v) else np.nan for v in errs]

styled = df.style.format({
    "vendor golden": "{:.3f}", "tier-2 cpwl": "{:.3f}",
    "error %": "{:+.2f}"}).background_gradient(
    subset=["error %"], cmap="RdYlGn_r", vmin=-40, vmax=40, na_rep="")
display(styled)
print()
print("VERDICT:", fit.verdict)
print("Evaluable:", fit.evaluable)
print("Waveform rel-RMSE: %.2f%%" % fit.waveform_rel_rmse_pct)